# Task 3 fine-tune -- Kaggle runner

1. **Targets**: synth data has no Task 3 labels, so Qwen2.5-Coder-14B writes Level 2/3
   feedback grounded on facts known by construction (injected bug, diff vs reference,
   oracle rubric, real g++ log). Level 1 and compile-error Level 2 are deterministic.
2. **Train**: QLoRA on Qwen2.5-Coder-7B (the model the pipeline actually uses), same
   self-diagnose prompt as inference, loss on the answer only.
3. **Eval**: base vs fine-tuned on the 32 gold `sample_dataset` items, same prompt, so
   the delta is the fine-tune alone.

**Needs accelerator "GPU T4 x2"** (Settings -> Accelerator) -- not settable via the push API.

In [ ]:
import torch
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() == 2, "set the accelerator to GPU T4 x2"


In [ ]:
%cd /kaggle/working
!rm -rf msc
!git clone --depth 1 https://github.com/ptran1203/msc.git
%cd /kaggle/working/msc/llm
!pip install -q -r requirements.txt
!which g++ || (apt-get update -qq && apt-get install -y -qq g++ > /dev/null)


In [ ]:
import shutil, pathlib, json
repo_llm = pathlib.Path("/kaggle/working/msc/llm")
PREV_TARGETS = "/kaggle/input/datasets/ptran1203/llm-task3-ft-targets"  # optional dataset with targets from an earlier run
shutil.copytree("/kaggle/input/datasets/ptran1203/llm-autograding-course-data/data", repo_llm / "data")
(repo_llm / "synth").mkdir(exist_ok=True)
shutil.copytree("/kaggle/input/notebooks/ptran1203/llm-synth-data-generator/synth_generated", repo_llm / "synth" / "generated")
n = json.loads((repo_llm / "synth/generated/task2_error_taxonomy.json").read_text())["n_samples"]
print("synth samples:", n)

# The mutator annotates its own edit in ~56% of samples ("// Sai == thành !=") -- the
# answer leaking into the input. Older synth outputs predate the in-generator fix.
import sys; sys.path.insert(0, str(repo_llm / "src"))
from synth_generate import strip_cpp_comments
n_stripped = 0
for p in (repo_llm / "synth/generated/submissions").rglob("*.cpp"):
    src = p.read_text(encoding="utf-8"); out = strip_cpp_comments(src)
    if out != src:
        p.write_text(out, encoding="utf-8"); n_stripped += 1
print("comments stripped from", n_stripped, "synth files")


## Smoke test first -- 2 targets + 2 optimizer steps

Per the project rule: fit a tiny batch end-to-end before the real run. Smoke outputs are
deleted afterwards so they don't end up in the real training set.

In [ ]:
%cd /kaggle/working/msc/llm/src
import os, subprocess, glob

def step(*args, gpu=0):
    # check=True: a failing `!python ...` does NOT stop a notebook, which once let a
    # broken train step slip past the smoke test and fail only hours later
    subprocess.run(["python", "-u", "finetune_task3.py", *args], env={**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}, check=True)

step("gen-targets", "--limit", "2")
print(open("../synth/generated/task3_ft_targets_shard0.jsonl", encoding="utf-8").read())
step("train", "--max-steps", "2")
for p in glob.glob("../synth/generated/task3_ft_targets_shard*.jsonl"):
    os.remove(p)
shutil.rmtree("../outputs/ft_task3_lora", ignore_errors=True)

# targets from an earlier run (private dataset), if attached -- gen-targets skips any
# sample_id already in its shard file, so only the missing ones get regenerated
for p in glob.glob(f"{PREV_TARGETS}/task3_ft_targets_shard*.jsonl"):
    shutil.copy(p, "../synth/generated/")
    print("reusing", p)


## 1. Generate targets -- one 14B teacher per GPU

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download("Qwen/Qwen2.5-Coder-14B-Instruct")  # once, so the 2 shards don't race on the cache
cmd = lambda i: ["python", "-u", "finetune_task3.py", "gen-targets", "--num-shards", "2", "--shard-id", str(i)]
procs = [subprocess.Popen(cmd(i), env={**os.environ, "CUDA_VISIBLE_DEVICES": str(i)}) for i in range(2)]
for p in procs:
    p.wait()
assert all(p.returncode == 0 for p in procs), [p.returncode for p in procs]
for p in sorted(glob.glob("../synth/generated/task3_ft_targets_shard*.jsonl")):
    print(p, sum(1 for _ in open(p, encoding="utf-8")), "targets")
# 14B weights (~30GB) aren't needed anymore -- free the disk before the 7B download
shutil.rmtree(os.path.expanduser("~/.cache/huggingface/hub/models--Qwen--Qwen2.5-Coder-14B-Instruct"), ignore_errors=True)


## 2. QLoRA train the 7B (single GPU)

In [ ]:
step("train")


## 3. Eval on the 32 gold samples -- base (GPU 0) vs fine-tuned (GPU 1) in parallel

In [ ]:
procs = [
    subprocess.Popen(["python", "-u", "finetune_task3.py", "eval", "--base"], env={**os.environ, "CUDA_VISIBLE_DEVICES": "0"}),
    subprocess.Popen(["python", "-u", "finetune_task3.py", "eval"], env={**os.environ, "CUDA_VISIBLE_DEVICES": "1"}),
]
for p in procs:
    p.wait()
assert all(p.returncode == 0 for p in procs), [p.returncode for p in procs]
for name in ("base_zeroshot", "finetuned"):
    print(json.loads(pathlib.Path(f"../outputs/ft_task3_eval/{name}.json").read_text())["metrics"])


## 4. Task 3 submission on public test (328) with the fine-tuned adapter

Runs through `run_prompting.py --adapter`: same zero-shot `task3_prompt` the adapter was
trained on (verified byte-identical), plus the production sanitize/retry/validate path.

In [ ]:
RUN_TAG = "qwen7b-ft-task3"
cmd = lambda i: ["python", "-u", "run_prompting.py", "--split", "public", "--tasks", "3", "--adapter", "../outputs/ft_task3_lora",
                 "--run-tag", RUN_TAG, "--num-shards", "2", "--shard-id", str(i)]
procs = [subprocess.Popen(cmd(i), env={**os.environ, "CUDA_VISIBLE_DEVICES": str(i)}) for i in range(2)]
for p in procs:
    p.wait()
assert all(p.returncode == 0 for p in procs), [p.returncode for p in procs]
subprocess.run(["python", "merge_shards.py", "--split", "public", "--tasks", "3", "--method-tag", RUN_TAG, "--num-shards", "2"], check=True)
subprocess.run(["python", "validate_submission.py", "3", f"../outputs/{RUN_TAG}/predictions_task3_public.json", "--split", "public"], check=True)


## Collect + clean up

Keeps only `/kaggle/working/ft_task3/` (targets, LoRA adapter, eval results, and
`submission/predictions_task3_finetuned.json`) and deletes
the cloned repo so the saved output stays small.

In [ ]:
out = pathlib.Path("/kaggle/working/ft_task3")
out.mkdir(exist_ok=True)
for p in glob.glob("../synth/generated/task3_ft_targets_shard*.jsonl"):
    shutil.copy(p, out)
shutil.copytree("../outputs/ft_task3_lora", out / "lora", ignore=shutil.ignore_patterns("checkpoints"), dirs_exist_ok=True)
shutil.copytree("../outputs/ft_task3_eval", out / "eval", dirs_exist_ok=True)
(out / "submission").mkdir(exist_ok=True)
shutil.copy(f"../outputs/{RUN_TAG}/predictions_task3_public.json", out / "submission" / "predictions_task3_finetuned.json")
%cd /kaggle/working
shutil.rmtree("/kaggle/working/msc", ignore_errors=True)
print(sorted(str(p.relative_to(out)) for p in out.rglob("*")))
